# Single-cell multi-omics analysis of the immune response in COVID-19
Wellcome HCA Strategic Science Support
Analysis of human blood immune cells provides insights into the coordinated response to viral infections such as severe acute respiratory syndrome coronavirus 2, which causes coronavirus disease 2019 (COVID-19). We performed single-cell transcriptome, surface proteome and T and B lymphocyte antigen receptor analyses of over 780,000 peripheral blood mononuclear cells from a cross-sectional cohort of 130 patients with varying severities of COVID-19. We identified expansion of nonclassical monocytes expressing complement transcripts (CD16+C1QA/B/C+) that sequester platelets and were predicted to replenish the alveolar macrophage pool in COVID-19. Early, uncommitted CD34+ hematopoietic stem/progenitor cells were primed toward megakaryopoiesis, accompanied by expanded megakaryocyte-committed progenitors and increased platelet activation. Clonally expanded CD8+ T cells and an increased ratio of CD8+ effector T cells to effector memory T cells characterized severe disease, while circulating follicular helper T cells accompanied mild disease. We observed a relative loss of IgA2 in symptomatic disease despite an overall expansion of plasmablasts and plasma cells. Our study highlights the coordinated immune response that contributes to COVID-19 pathogenesis and reveals discrete cellular components that can be targeted for therapy.



https://www.nature.com/articles/s41591-021-01329-2

https://cellxgene.cziscience.com/collections/ddfad306-714d-4cc0-9985-d9072820c530

In [1]:
import scanpy as sc
import polars as pl
import os

In [2]:
data_dir = "/home/rstudio/project/XX_exam_data/data"
input_dir = os.path.join(data_dir, "haniffa_stevenson_covid19.h5ad")

In [3]:
adata = sc.read_h5ad(input_dir)

In [4]:
adata

AnnData object with n_obs × n_vars = 647366 × 24245
    obs: 'sample_id', 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'initial_clustering', 'Resample', 'Collection_Day', 'Swab_result', 'Status', 'Smoker', 'Status_on_day_collection', 'Status_on_day_collection_summary', 'Days_from_onset', 'Site', 'time_after_LPS', 'Worst_Clinical_Status', 'Outcome', 'donor_id', 'assay_ontology_term_id', 'cell_type_ontology_term_id', 'development_stage_ontology_term_id', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'is_primary_data', 'sex_ontology_term_id', 'tissue_ontology_term_id', 'author_cell_type', 'suspension_type', 'tissue_type', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid'
    var: 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type'
    uns: 'antibody_X', 'antibody_features', 'antibody_raw.X', 'citat

In [5]:
import numpy as np
import scipy.sparse as sp

X = adata.X
total_counts = adata.obs["total_counts"].to_numpy()
target_sum = 1e4

if sp.issparse(X):
    X_norm = X.copy()
    X_norm.data = np.expm1(X_norm.data)
    raw_approx = X_norm.multiply(total_counts[:, None] / target_sum)
    raw_approx.data = np.rint(raw_approx.data)
else:
    X_norm = np.expm1(X)
    raw_approx = np.rint(X_norm * (total_counts[:, None] / target_sum)).astype(int)

In [6]:
adata.layers["counts"] = raw_approx.tocsr()

In [7]:
adata.uns["antibody_raw.X"].shape

(647366, 192)

In [8]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
import scanpy as sc
from anndata import AnnData
from geosketch import gs


def geosketch_rna_adt_from_uns(
    adata,
    n_cells: int = 10_000,
    rep: str = "X_pca",
    seed: int = 1,
    adt_counts_key: str = "antibody_raw.X",
    adt_features_key: str = "antibody_features",
    sketch_key: str = "is_geosketch",
):
    """
    Geosketch cells from an AnnData object using an embedding in adata.obsm,
    while manually subsetting ADT counts stored in adata.uns.

    Returns
    -------
    rna_sketch
        AnnData object with RNA data.
    adt_sketch
        AnnData object with ADT counts.
    sketch_idx
        Integer indices of selected cells.
    """
    if rep not in adata.obsm:
        raise KeyError(f"{rep!r} not found in adata.obsm")

    if adt_counts_key not in adata.uns:
        raise KeyError(f"{adt_counts_key!r} not found in adata.uns")

    if adt_features_key not in adata.uns:
        raise KeyError(f"{adt_features_key!r} not found in adata.uns")

    if n_cells > adata.n_obs:
        raise ValueError(
            f"n_cells={n_cells} is larger than adata.n_obs={adata.n_obs}"
        )

    np.random.seed(seed)

    # 1. Select representative cells using RNA PCA or another embedding
    X_rep = adata.obsm[rep]
    sketch_idx = gs(X_rep, n_cells, replace=False)

    # Make ordering stable/reproducible in the output object
    sketch_idx = np.asarray(sketch_idx)
    sketch_idx = np.sort(sketch_idx)

    # 2. Mark selected cells in the full object
    adata.obs[sketch_key] = False
    adata.obs.loc[adata.obs_names[sketch_idx], sketch_key] = True

    # 3. Subset RNA normally
    rna_sketch = adata[sketch_idx].copy()

    # Optional: remove the bulky ADT objects from RNA .uns
    # because we will save ADT as its own object.
    for key in [adt_counts_key, "antibody_X", adt_features_key]:
        if key in rna_sketch.uns:
            del rna_sketch.uns[key]

    # 4. Manually subset ADT counts stored in .uns
    adt_counts = adata.uns[adt_counts_key]

    if sp.issparse(adt_counts):
        adt_counts_sketch = adt_counts[sketch_idx, :].copy()
    else:
        adt_counts_sketch = np.asarray(adt_counts)[sketch_idx, :].copy()

    # 5. Build ADT feature metadata
    adt_features = adata.uns[adt_features_key]

    if isinstance(adt_features, pd.DataFrame):
        adt_var = adt_features.copy()
        if adt_var.index.size != adt_counts_sketch.shape[1]:
            raise ValueError(
                "antibody_features has a different number of rows than ADT features."
            )
    else:
        adt_names = pd.Index(np.asarray(adt_features).astype(str), name="feature_name")
        adt_var = pd.DataFrame(index=adt_names)

    # Ensure ADT var names are unique
    adt_var.index = adt_var.index.astype(str)
    adt_var_names = pd.Index(adt_var.index)
    if not adt_var_names.is_unique:
        adt_var.index = pd.Index(
            [f"{name}_{i}" for i, name in enumerate(adt_var.index)],
            name=adt_var.index.name,
        )

    # 6. Use same obs as RNA sketch
    adt_obs = rna_sketch.obs.copy()

    adt_sketch = AnnData(
        X=adt_counts_sketch,
        obs=adt_obs,
        var=adt_var,
    )

    adt_sketch.var["feature_type"] = "Antibody Capture"

    return rna_sketch, adt_sketch, sketch_idx

In [9]:
rna_sketch, adt_sketch, sketch_idx = geosketch_rna_adt_from_uns(
    adata,
    n_cells=30_000,
    rep="X_pca",
    seed=1,
)

In [10]:
rna_sketch

AnnData object with n_obs × n_vars = 30000 × 24245
    obs: 'sample_id', 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'initial_clustering', 'Resample', 'Collection_Day', 'Swab_result', 'Status', 'Smoker', 'Status_on_day_collection', 'Status_on_day_collection_summary', 'Days_from_onset', 'Site', 'time_after_LPS', 'Worst_Clinical_Status', 'Outcome', 'donor_id', 'assay_ontology_term_id', 'cell_type_ontology_term_id', 'development_stage_ontology_term_id', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'is_primary_data', 'sex_ontology_term_id', 'tissue_ontology_term_id', 'author_cell_type', 'suspension_type', 'tissue_type', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid', 'is_geosketch'
    var: 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type'
    uns: 'citation', 'default_embedding', 'hvg', 'lei

In [11]:
adt_sketch.var

,antibody_features,feature_type
AB_CD80,CD80,Antibody Capture
AB_CD86,CD86,Antibody Capture
AB_CD274,CD274,Antibody Capture
AB_PDCD1LG2,PDCD1LG2,Antibody Capture
AB_ICOSLG,ICOSLG,Antibody Capture
...,...,...
AB_Podocalyxin,Podocalyxin,Antibody Capture
AB_GGT1,GGT1,Antibody Capture
AB_c-Met,c-Met,Antibody Capture
AB_LIGHT,LIGHT,Antibody Capture


In [12]:
import anndata as ad

ad.settings.allow_write_nullable_strings = True

In [13]:
rna_sketch.write_h5ad(
    os.path.join(data_dir, "exam_rna_geosketch_30k.h5ad"),
    compression="gzip",
)

adt_sketch.write_h5ad(
    os.path.join(data_dir, "exam_adt_geosketch_30k.h5ad"),
    compression="gzip",
)